# Biased Matrix Factorization 

In this notebook, we create from scratch a BMF algorithm, train it and validate it on the given dataset. The target is to port later the validated algorithm into our application backend for our users.  

Since the training dataset is not matching our database, we need this notebook to develop correctly our algorithm and then plug it into our project.

---

### The Algorithm

For a user $u$ and an event $i$, the predicted rating is:

$$\hat{r}_{ui} = \mu + b_u + b_i + \mathbf{p}_u^\top \mathbf{q}_i$$

- $\mu$ — global mean rating
- $b_u$ — user bias (how positive/negative this user tends to be)
- $b_i$ — item bias (how popular this event is overall)
- $\mathbf{p}_u \in \mathbb{R}^k$ — user latent vector
- $\mathbf{q}_i \in \mathbb{R}^k$ — item latent vector

We minimise the **regularised squared error** over observed ratings:

$$\min_{b_*, \mathbf{p}_*, \mathbf{q}_*} \sum_{(u,i)\in\mathcal{K}} \left(r_{ui} - \hat{r}_{ui}\right)^2
+ \lambda\left(b_u^2 + b_i^2 + \lVert\mathbf{p}_u\rVert^2 + \lVert\mathbf{q}_i\rVert^2\right)$$

trained with **stochastic gradient descent**.

## Set the environment

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RNG_SEED = 42
np.random.seed(RNG_SEED)

# >>> EDIT THIS to point to the folder that holds the CSV files <<<
DATA_DIR = r"C:\Users\oiko\Desktop\DIT\Web_Applications\web_app\dataset\rel_event_csvs"

import os
print("Files in dataset folder:")
for f in os.listdir(DATA_DIR):
    size_mb = os.path.getsize(os.path.join(DATA_DIR, f)) / 1e6
    print(f"  {f:<25} {size_mb:8.1f} MB")

Files in dataset folder:
  events.csv                   641.6 MB
  event_attendees.csv          407.2 MB
  event_interest.csv             0.6 MB
  users.csv                      2.6 MB
  user_friends.csv             466.8 MB


## Load the training Dataset

In [9]:
event_interest = pd.read_csv(os.path.join(DATA_DIR, "event_interest.csv"))
print("Shape: ", event_interest.shape)
event_interest.head()

Shape:  (14978, 6)


,user,event,invited,timestamp,interested,not_interested
0,8949,1130067.0,0,2012-10-02 15:53:05.754,0,0
1,8949,1130381.0,0,2012-10-02 15:53:05.754,0,0
2,8949,1566551.0,0,2012-10-02 15:53:05.754,1,0
3,8949,1184649.0,0,2012-10-02 15:53:05.754,0,0
4,8949,1182052.0,0,2012-10-02 15:53:05.754,0,0


## Clear Dataset  

|           Signal         |      Rating         |
| :----------------------: | :-----------------: |
| `interested = 1`         |  **1.0** (positive) |
| `not_interested = 1`     |  **0.0** (negative) |
| both `0`                 |    *dropped*        |

In [11]:
def build_ratings(df):
    rows = []
    for user, event, interested, not_interested in zip(
        df["user"], df["event"], df["interested"], df["not_interested"]
    ):
        if interested == 1:
            rows.append((user, event, 1.0))
        elif not_interested == 1:
            rows.append((user, event, 0.0))
        # both 0 -> no real opinion
    return pd.DataFrame(rows, columns=["user", "event", "rating"])

ratings = build_ratings(event_interest)
print("explicit ratings:", len(ratings))
print("unique users :", ratings["user"].nunique())
print("unique events:", ratings["event"].nunique())
print("positive ratio:", round((ratings.rating == 1.0).mean(), 3))
ratings.head()

explicit ratings: 4547
unique users : 1979
unique events: 2739
positive ratio: 0.888


,user,event,rating
0,8949,1566551.0,1.0
1,20546,1806207.0,1.0
2,20546,1709580.0,1.0
3,20546,1751156.0,1.0
4,23668,NaN,1.0


## Enrich Ratings with Attendance

In [12]:
USE_ATTENDEES = True

if USE_ATTENDEES:
    status_to_rating = {"yes": 1.0, "maybe": 0.6, "no": 0.0}
    extra = []
    path = os.path.join(DATA_DIR, "event_attendees.csv")
    for chunk in pd.read_csv(path, usecols=["event", "status", "user_id"], chunksize=500_000):
        chunk = chunk.dropna(subset=["user_id"])
        chunk = chunk[chunk["status"].isin(status_to_rating)]
        for u, e, s in zip(chunk["user_id"], chunk["event"], chunk["status"]):
            extra.append((int(u), int(e), status_to_rating[s]))
    extra_df = pd.DataFrame(extra, columns=["user", "event", "rating"])
    print("attendance ratings:", len(extra_df))
    # combine; if a (user,event) appears in both, keep the explicit one (interest)
    ratings = pd.concat([ratings, extra_df]).drop_duplicates(
        subset=["user", "event"], keep="first"
    ).reset_index(drop=True)
    print("combined ratings:", len(ratings))

attendance ratings: 10392
combined ratings: 14858
